In [1]:
import folium
from folium.plugins import MarkerCluster
import json
import pandas as pd
import html as html_module
import re
from math import radians, cos, sin, sqrt
from pathlib import Path
from IPython.display import display, HTML

# =========================================================
# CONFIG — chemins relatifs, rien à modifier
# =========================================================
BASE_DIR    = Path(".")
TEXTS_DIR   = BASE_DIR / "data_texts"
FR_PATH     = TEXTS_DIR / "20000lieues_fr.txt"
EN_PATH     = TEXTS_DIR / "20000lieues_an.txt"
JSON_PATH   = BASE_DIR / "escales_enrichies_dates_corrigees.json"
OUTPUT_HTML = BASE_DIR / "CIAVisualisationnautilusmultilang_fixed.html"

for p in [TEXTS_DIR, FR_PATH, EN_PATH, JSON_PATH]:
    assert p.exists(), f"❌ Introuvable : {p.resolve()}"
    print(f"✅ {p.resolve()}")

fr_text_full = FR_PATH.read_text(encoding="utf-8", errors="ignore")
en_text_full = EN_PATH.read_text(encoding="utf-8", errors="ignore")
print(f"📖 FR : {len(fr_text_full):,} caractères")
print(f"📖 EN : {len(en_text_full):,} caractères")

with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)
print(f"🗺️  {len(data)} escales chargées")

# =========================================================
# FONCTIONS UTILITAIRES
# =========================================================
def clean_text(txt):
    if txt is None:
        return ""
    txt = str(txt)
    txt = txt.replace("\u00a0", " ").replace("\ufeff", "")
    txt = re.sub(r"\r\n?", "\n", txt)
    txt = re.sub(r"[ \t]+", " ", txt)
    txt = re.sub(r"\n{3,}", "\n\n", txt)
    return txt.strip()

def get_lang(value, lang):
    """Lit la valeur FR ou EN depuis un champ bilingue du JSON."""
    if isinstance(value, dict):
        v = value.get(lang) or value.get("fr") or ""
        return clean_text(str(v)) if v else ""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    return clean_text(str(value))

def get_passage(row, lang):
    """Lit passage_roman.fr ou passage_roman.en depuis le JSON v3."""
    passage = row.get("passage_roman", None)
    if isinstance(passage, dict):
        val = passage.get(lang)
        if val and str(val).strip() not in ("", "None", "null"):
            return clean_text(str(val))
        return ""
    if passage and isinstance(passage, str):
        return clean_text(passage)
    return ""

def distance_from_center(lat):
    R = 6371e3
    lat_rad = radians(lat)
    return int(sqrt((R * cos(lat_rad))**2 + (R * sin(lat_rad))**2))

def make_html_content(nom, texte, label_titre):
    """Formate un extrait en HTML pour la Bootstrap Modal."""
    paras = [p.strip() for p in texte.replace("\n\n", "\n").split("\n") if p.strip()]
    paras_html = "".join(f"<p style='margin-bottom:14px; line-height:1.8; font-size:1.2rem;'>{p}</p>" for p in paras[:4])
    return f"""
        <h3 style='color:#0a3d62; font-size:1.3rem; margin-bottom:8px;'>{nom}</h3>
        <h5 style='color:#27c39f; font-size:1rem; margin-bottom:16px; font-weight:600;'>{label_titre}</h5>
        {paras_html}"""

# =========================================================
# DATAFRAME
# =========================================================
df = pd.DataFrame(data)
df["Latitude_Décimal"]  = pd.to_numeric(df["Latitude_Décimal"],  errors="coerce")
df["Longitude_Décimal"] = pd.to_numeric(df["Longitude_Décimal"], errors="coerce")
df = df.dropna(subset=["Latitude_Décimal", "Longitude_Décimal"]).reset_index(drop=True)
df["DistanceCentreTerre"] = df["Latitude_Décimal"].apply(distance_from_center)
print(f"✅ {len(df)} escales avec coordonnées valides")

# =========================================================
# FICHE 2 — EXTRACTION DES ESPÈCES MARINES PAR CHAPITRES
# Méthode : NLP regex + dictionnaire exhaustif (source : texte JV + INPN)
# Résultat : especes calculées à la volée depuis les vrais passages du livre
# =========================================================

import re as _re

# ── Dictionnaire exhaustif FR → { emoji, cat, fr, en } ──
_ESPECES_DB = {
    "baleine":        {"emoji": "🐋", "cat": "Cétacé",      "fr": "Baleine",         "en": "Whale"},
    "cachalot":       {"emoji": "🐋", "cat": "Cétacé",      "fr": "Cachalot",        "en": "Sperm Whale"},
    "dauphin":        {"emoji": "🐬", "cat": "Cétacé",      "fr": "Dauphin",         "en": "Dolphin"},
    "marsouin":       {"emoji": "🐬", "cat": "Cétacé",      "fr": "Marsouin",        "en": "Porpoise"},
    "narwal":         {"emoji": "🦷", "cat": "Cétacé",      "fr": "Narval",          "en": "Narwhal"},
    "narval":         {"emoji": "🦷", "cat": "Cétacé",      "fr": "Narval",          "en": "Narwhal"},
    "orque":          {"emoji": "🐋", "cat": "Cétacé",      "fr": "Orque",           "en": "Orca"},
    "cétacé":         {"emoji": "🐋", "cat": "Cétacé",      "fr": "Cétacé",          "en": "Cetacean"},
    "rorqual":        {"emoji": "🐋", "cat": "Cétacé",      "fr": "Rorqual",         "en": "Rorqual"},
    "jubarte":        {"emoji": "🐋", "cat": "Cétacé",      "fr": "Jubarte",         "en": "Humpback Whale"},
    "baleinoptère":   {"emoji": "🐋", "cat": "Cétacé",      "fr": "Baleinoptère",    "en": "Fin Whale"},
    "requin":         {"emoji": "🦈", "cat": "Poisson",     "fr": "Requin",          "en": "Shark"},
    "squale":         {"emoji": "🦈", "cat": "Poisson",     "fr": "Squale",          "en": "Shark"},
    "raie":           {"emoji": "🐟", "cat": "Poisson",     "fr": "Raie",            "en": "Ray"},
    "thon":           {"emoji": "🐟", "cat": "Poisson",     "fr": "Thon",            "en": "Tuna"},
    "saumon":         {"emoji": "🐟", "cat": "Poisson",     "fr": "Saumon",          "en": "Salmon"},
    "morue":          {"emoji": "🐟", "cat": "Poisson",     "fr": "Morue",           "en": "Cod"},
    "hareng":         {"emoji": "🐟", "cat": "Poisson",     "fr": "Hareng",          "en": "Herring"},
    "espadon":        {"emoji": "🐟", "cat": "Poisson",     "fr": "Espadon",         "en": "Swordfish"},
    "murène":         {"emoji": "🐟", "cat": "Poisson",     "fr": "Murène",          "en": "Moray"},
    "lamproie":       {"emoji": "🐟", "cat": "Poisson",     "fr": "Lamproie",        "en": "Lamprey"},
    "dorade":         {"emoji": "🐟", "cat": "Poisson",     "fr": "Dorade",          "en": "Sea Bream"},
    "turbot":         {"emoji": "🐟", "cat": "Poisson",     "fr": "Turbot",          "en": "Turbot"},
    "grondin":        {"emoji": "🐟", "cat": "Poisson",     "fr": "Grondin",         "en": "Gurnard"},
    "baudroie":       {"emoji": "🐟", "cat": "Poisson",     "fr": "Baudroie",        "en": "Anglerfish"},
    "hippocampe":     {"emoji": "🐴", "cat": "Poisson",     "fr": "Hippocampe",      "en": "Seahorse"},
    "rémora":         {"emoji": "🐟", "cat": "Poisson",     "fr": "Rémora",          "en": "Remora"},
    "orphie":         {"emoji": "🐟", "cat": "Poisson",     "fr": "Orphie",          "en": "Garfish"},
    "labre":          {"emoji": "🐟", "cat": "Poisson",     "fr": "Labre",           "en": "Wrasse"},
    "spare":          {"emoji": "🐟", "cat": "Poisson",     "fr": "Spare",           "en": "Sparidae"},
    "sciène":         {"emoji": "🐟", "cat": "Poisson",     "fr": "Sciène",          "en": "Croaker"},
    "coriphène":      {"emoji": "🐟", "cat": "Poisson",     "fr": "Coriphène",       "en": "Mahi-mahi"},
    "blémie":         {"emoji": "🐟", "cat": "Poisson",     "fr": "Blémie",          "en": "Blenny"},
    "gobie":          {"emoji": "🐟", "cat": "Poisson",     "fr": "Gobie",           "en": "Goby"},
    "centronote":     {"emoji": "🐟", "cat": "Poisson",     "fr": "Centronote",      "en": "Pilot fish"},
    "mugilomor":      {"emoji": "🐟", "cat": "Poisson",     "fr": "Mugilomor",       "en": "Mullet"},
    "pintadine":      {"emoji": "🐟", "cat": "Poisson",     "fr": "Pintadine",       "en": "Pearl oyster fish"},
    "chondropté":     {"emoji": "🦈", "cat": "Poisson",     "fr": "Chondroptérygien","en": "Cartilaginous fish"},
    "poulpe":         {"emoji": "🐙", "cat": "Mollusque",   "fr": "Poulpe",          "en": "Octopus"},
    "pieuvre":        {"emoji": "🐙", "cat": "Mollusque",   "fr": "Pieuvre",         "en": "Octopus"},
    "calmar":         {"emoji": "🦑", "cat": "Mollusque",   "fr": "Calmar",          "en": "Squid"},
    "seiche":         {"emoji": "🦑", "cat": "Mollusque",   "fr": "Seiche",          "en": "Cuttlefish"},
    "nautile":        {"emoji": "🐚", "cat": "Mollusque",   "fr": "Nautile",         "en": "Nautilus"},
    "huître":         {"emoji": "🦪", "cat": "Mollusque",   "fr": "Huître",          "en": "Oyster"},
    "moule":          {"emoji": "🦪", "cat": "Mollusque",   "fr": "Moule",           "en": "Mussel"},
    "pétoncle":       {"emoji": "🦪", "cat": "Mollusque",   "fr": "Pétoncle",        "en": "Scallop"},
    "buccin":         {"emoji": "🐚", "cat": "Mollusque",   "fr": "Buccin",          "en": "Whelk"},
    "triton":         {"emoji": "🐚", "cat": "Mollusque",   "fr": "Triton",          "en": "Triton shell"},
    "volute":         {"emoji": "🐚", "cat": "Mollusque",   "fr": "Volute",          "en": "Volute"},
    "murex":          {"emoji": "🐚", "cat": "Mollusque",   "fr": "Murex",           "en": "Murex"},
    "tridacne":       {"emoji": "🦪", "cat": "Mollusque",   "fr": "Tridacne",        "en": "Giant clam"},
    "ptéropode":      {"emoji": "🐚", "cat": "Mollusque",   "fr": "Ptéropode",       "en": "Sea butterfly"},
    "coquillage":     {"emoji": "🐚", "cat": "Mollusque",   "fr": "Coquillage",      "en": "Shell"},
    "aplysie":        {"emoji": "🐚", "cat": "Mollusque",   "fr": "Aplysie",         "en": "Sea hare"},
    "aplysis":        {"emoji": "🐚", "cat": "Mollusque",   "fr": "Aplysie",         "en": "Sea hare"},
    "bivalve":        {"emoji": "🦪", "cat": "Mollusque",   "fr": "Bivalve",         "en": "Bivalve"},
    "testacé":        {"emoji": "🐚", "cat": "Mollusque",   "fr": "Testacé",         "en": "Testacean"},
    "gastéropode":    {"emoji": "🐚", "cat": "Mollusque",   "fr": "Gastéropode",     "en": "Gastropod"},
    "meleagrina":     {"emoji": "🦪", "cat": "Mollusque",   "fr": "Meleagrina",      "en": "Pearl oyster"},
    "homard":         {"emoji": "🦞", "cat": "Crustacé",   "fr": "Homard",          "en": "Lobster"},
    "langouste":      {"emoji": "🦞", "cat": "Crustacé",   "fr": "Langouste",       "en": "Spiny lobster"},
    "languste":       {"emoji": "🦞", "cat": "Crustacé",   "fr": "Langouste",       "en": "Spiny lobster"},
    "crabe":          {"emoji": "🦀", "cat": "Crustacé",   "fr": "Crabe",           "en": "Crab"},
    "crevette":       {"emoji": "🦐", "cat": "Crustacé",   "fr": "Crevette",        "en": "Shrimp"},
    "cirrhipède":     {"emoji": "🦐", "cat": "Crustacé",   "fr": "Cirrhipède",      "en": "Barnacle"},
    "stomapode":      {"emoji": "🦐", "cat": "Crustacé",   "fr": "Stomapode",       "en": "Mantis shrimp"},
    "oursin":         {"emoji": "🦔", "cat": "Échinoderme","fr": "Oursin",           "en": "Sea urchin"},
    "holothurie":     {"emoji": "🐛", "cat": "Échinoderme","fr": "Holothurie",       "en": "Sea cucumber"},
    "astérie":        {"emoji": "⭐", "cat": "Échinoderme","fr": "Astérie",          "en": "Starfish"},
    "encrine":        {"emoji": "🌿", "cat": "Échinoderme","fr": "Encrine",          "en": "Sea lily"},
    "crinoïde":       {"emoji": "🌿", "cat": "Échinoderme","fr": "Crinoïde",         "en": "Crinoid"},
    "ophiure":        {"emoji": "⭐", "cat": "Échinoderme","fr": "Ophiure",          "en": "Brittle star"},
    "pentacrine":     {"emoji": "🌿", "cat": "Échinoderme","fr": "Pentacrine",       "en": "Pentacrinus"},
    "troque":         {"emoji": "🐚", "cat": "Échinoderme","fr": "Troque",           "en": "Top shell"},
    "fissurelle":     {"emoji": "🐚", "cat": "Mollusque",  "fr": "Fissurelle",       "en": "Keyhole limpet"},
    "méduse":         {"emoji": "🪼", "cat": "Cnidaire",   "fr": "Méduse",           "en": "Jellyfish"},
    "physalie":       {"emoji": "🪼", "cat": "Cnidaire",   "fr": "Physalie",         "en": "Portuguese man o' war"},
    "anémone":        {"emoji": "🌸", "cat": "Cnidaire",   "fr": "Anémone de mer",   "en": "Sea anemone"},
    "corail":         {"emoji": "🪸", "cat": "Cnidaire",   "fr": "Corail",           "en": "Coral"},
    "madrépore":      {"emoji": "🪸", "cat": "Cnidaire",   "fr": "Madrépore",        "en": "Madrepore"},
    "gorgone":        {"emoji": "🪸", "cat": "Cnidaire",   "fr": "Gorgone",          "en": "Gorgonian"},
    "alcyon":         {"emoji": "🪸", "cat": "Cnidaire",   "fr": "Alcyon",           "en": "Soft coral"},
    "polype":         {"emoji": "🪸", "cat": "Cnidaire",   "fr": "Polype",           "en": "Polyp"},
    "zoophyte":       {"emoji": "🌿", "cat": "Cnidaire",   "fr": "Zoophyte",         "en": "Zoophyte"},
    "actinie":        {"emoji": "🌸", "cat": "Cnidaire",   "fr": "Actinie",          "en": "Sea anemone"},
    "tubulaire":      {"emoji": "🌿", "cat": "Cnidaire",   "fr": "Tubulaire",        "en": "Tubularia"},
    "coralligèn":     {"emoji": "🪸", "cat": "Cnidaire",   "fr": "Formation coralligène","en": "Coral formation"},
    "morse":          {"emoji": "🦭", "cat": "Mammifère",  "fr": "Morse",            "en": "Walrus"},
    "phoque":         {"emoji": "🦭", "cat": "Mammifère",  "fr": "Phoque",           "en": "Seal"},
    "lamantin":       {"emoji": "🐋", "cat": "Mammifère",  "fr": "Lamantin",         "en": "Manatee"},
    "dugong":         {"emoji": "🐋", "cat": "Mammifère",  "fr": "Dugong",           "en": "Dugong"},
    "otarie":         {"emoji": "🦭", "cat": "Mammifère",  "fr": "Otarie",           "en": "Sea lion"},
    "tortue":         {"emoji": "🐢", "cat": "Reptile",    "fr": "Tortue marine",    "en": "Sea turtle"},
    "albatros":       {"emoji": "🐦", "cat": "Oiseau",     "fr": "Albatros",         "en": "Albatross"},
    "pingouin":       {"emoji": "🐧", "cat": "Oiseau",     "fr": "Pingouin",         "en": "Penguin"},
    "manchot":        {"emoji": "🐧", "cat": "Oiseau",     "fr": "Manchot",          "en": "Penguin"},
    "pétrel":         {"emoji": "🐦", "cat": "Oiseau",     "fr": "Pétrel",           "en": "Petrel"},
    "mouette":        {"emoji": "🐦", "cat": "Oiseau",     "fr": "Mouette",          "en": "Seagull"},
    "algue":          {"emoji": "🌿", "cat": "Algue",      "fr": "Algue",            "en": "Algae"},
    "varech":         {"emoji": "🌿", "cat": "Algue",      "fr": "Varech",           "en": "Kelp"},
    "sargasse":       {"emoji": "🌿", "cat": "Algue",      "fr": "Sargasse",         "en": "Sargasso"},
    "fucus":          {"emoji": "🌿", "cat": "Algue",      "fr": "Fucus",            "en": "Fucus"},
    "laminaire":      {"emoji": "🌿", "cat": "Algue",      "fr": "Laminaire",        "en": "Kelp"},
    "posidonie":      {"emoji": "🌿", "cat": "Algue",      "fr": "Posidonie",        "en": "Posidonia"},
    "zostère":        {"emoji": "🌿", "cat": "Algue",      "fr": "Zostère",          "en": "Seagrass"},
    "hydrophyte":     {"emoji": "🌿", "cat": "Algue",      "fr": "Hydrophyte",       "en": "Aquatic plant"},
    "thalassiophyte": {"emoji": "🌿", "cat": "Algue",      "fr": "Thalassiophyte",   "en": "Marine alga"},
    "éponge":         {"emoji": "🧽", "cat": "Éponge",     "fr": "Éponge",           "en": "Sponge"},
    "spongiaire":     {"emoji": "🧽", "cat": "Éponge",     "fr": "Spongiaire",       "en": "Sponge"},
    "annélide":       {"emoji": "🪱", "cat": "Ver marin",  "fr": "Annélide",         "en": "Annelid"},
}

# ── Correspondance mots EN → clé FR ─────────────────────
_EN_TO_FR = {
    "whale": "baleine", "sperm whale": "cachalot", "dolphin": "dauphin",
    "porpoise": "marsouin", "narwhal": "narval", "narwhale": "narval",
    "orca": "orque", "killer whale": "orque", "rorqual": "rorqual",
    "humpback": "jubarte", "shark": "requin", "ray": "raie", "tuna": "thon",
    "salmon": "saumon", "cod": "morue", "herring": "hareng", "swordfish": "espadon",
    "moray": "murène", "seahorse": "hippocampe", "octopus": "poulpe",
    "poulp": "poulpe", "squid": "calmar", "cuttlefish": "seiche",
    "nautilus": "nautile", "oyster": "huître", "mussel": "moule",
    "scallop": "pétoncle", "clam": "tridacne", "lobster": "homard",
    "crab": "crabe", "shrimp": "crevette", "barnacle": "cirrhipède",
    "sea urchin": "oursin", "starfish": "astérie", "sea cucumber": "holothurie",
    "sea lily": "encrine", "jellyfish": "méduse", "medusa": "méduse",
    "medusae": "méduse", "portuguese man": "physalie", "sea anemone": "anémone",
    "anemone": "anémone", "coral": "corail", "gorgonian": "gorgone",
    "polyp": "polype", "walrus": "morse", "seal": "phoque", "manatee": "lamantin",
    "dugong": "dugong", "sea lion": "otarie", "turtle": "tortue",
    "albatross": "albatros", "penguin": "pingouin", "petrel": "pétrel",
    "seagull": "mouette", "algae": "algue", "kelp": "laminaire",
    "sargasso": "sargasse", "sargassum": "sargasse", "seagrass": "zostère",
    "sponge": "éponge", "annelid": "annélide", "zoophyte": "zoophyte",
    "fucus": "fucus", "sea spider": "crabe",
}

# ── Découpage du texte en chapitres ─────────────────────
def _split_chapitres(texte):
    chapitres = {}
    pattern = r'(CHAPITRE\s+(?:PREMIER|[IVXLCDM]+))\s*\r?\n'
    matches = list(_re.finditer(pattern, texte))
    for i, m in enumerate(matches):
        nom = _re.sub(r'\s+', ' ', m.group(1)).strip()
        debut = m.end()
        fin = matches[i+1].start() if i+1 < len(matches) else len(texte)
        chapitres[nom] = texte[debut:fin]
    return chapitres

# Pré-calculer les chapitres une seule fois
_premiers = [m.start() for m in _re.finditer(r'CHAPITRE PREMIER', fr_text_full)]
_fr_p1 = fr_text_full[:_premiers[1]] if len(_premiers) >= 2 else fr_text_full
_fr_p2 = fr_text_full[_premiers[1]:] if len(_premiers) >= 2 else ""
_ch_fr_p1 = _split_chapitres(_fr_p1)
_ch_fr_p2 = _split_chapitres(_fr_p2)
_ch_en     = _split_chapitres(en_text_full)

# ── Mapping escale idx → chapitres ──────────────────────
_NUMERAUX = {
    "PREMIER": "CHAPITRE PREMIER",
    **{r: f"CHAPITRE {r}" for r in [
        "I","II","III","IV","V","VI","VII","VIII","IX","X",
        "XI","XII","XIII","XIV","XV","XVI","XVII","XVIII","XIX","XX",
        "XXI","XXII","XXIII","XXIV"
    ]}
}

_ESCALE_CHAPITRES = {
    # idx: (chapitres_fr_partie1, chapitres_fr_partie2, chapitres_en)
    0:  (["PREMIER","II"],      [],                      ["IX","X"]),
    1:  ([],                    [],                      ["XVIII"]),
    2:  ([],                    [],                      ["XIX","XX"]),
    3:  ([],                    [],                      ["XX","XXI"]),
    4:  ([],                    [],                      ["XXI"]),
    5:  ([],                    [],                      ["XXI","XXII"]),
    6:  ([],                    [],                      ["XXII","XXIII"]),
    7:  ([],                    [],                      ["XXIII","XXIV"]),
    8:  ([],                    [],                      []),
    9:  ([],                    ["I","II"],               ["I","II"]),
    10: ([],                    ["V","VI"],               ["V","VI"]),
    11: ([],                    ["VII"],                  ["VII"]),
    12: ([],                    ["IX","X"],               ["IX","X"]),
    13: ([],                    ["XI"],                   ["XI"]),
    14: ([],                    [],                      []),
    15: ([],                    [],                      []),
    16: ([],                    ["XIV"],                  ["XIV"]),
    17: ([],                    ["XIV","XV","XVI"],       ["XIV","XV","XVI"]),
    18: ([],                    ["XVII"],                 ["XVII"]),
    19: ([],                    [],                      []),
    20: ([],                    [],                      []),
    21: ([],                    ["XVIII"],                ["XVIII"]),
    22: ([],                    [],                      []),
    23: ([],                    ["XIX","XX"],             ["XIX","XX"]),
    24: ([],                    ["XX"],                   ["XX"]),
    25: ([],                    ["XX","XXI"],             ["XX","XXI"]),
    26: ([],                    ["XXI"],                  ["XXI"]),
    27: ([],                    ["XXII"],                 ["XXII"]),
    28: ([],                    ["XXII","XXIII"],         ["XXII","XXIII"]),
    29: ([],                    [],                      []),
    30: ([],                    [],                      []),
    31: ([],                    ["XXIII"],                ["XXIII"]),
}

def _get_passage_chapitres(idx):
    """Retourne (texte_fr, texte_en) pour une escale donnée."""
    ch_p1, ch_p2, ch_en_list = _ESCALE_CHAPITRES.get(idx, ([], [], []))
    parts_fr = []
    for r in ch_p1:
        k = _NUMERAUX.get(r, f"CHAPITRE {r}")
        if k in _ch_fr_p1:
            parts_fr.append(_ch_fr_p1[k])
    for r in ch_p2:
        k = _NUMERAUX.get(r, f"CHAPITRE {r}")
        if k in _ch_fr_p2:
            parts_fr.append(_ch_fr_p2[k])
    parts_en = []
    for r in ch_en_list:
        k = _NUMERAUX.get(r, f"CHAPITRE {r}")
        if k in _ch_en:
            parts_en.append(_ch_en[k])
    return " ".join(parts_fr), " ".join(parts_en)

def _matcher_fr(texte):
    """Retourne dict {cle: info} des espèces trouvées dans un texte FR."""
    found = {}
    tl = texte.lower()
    for mot, info in _ESPECES_DB.items():
        if _re.search(r'\b' + _re.escape(mot.lower()) + r'[a-zéèêëàâùûüîïôœç]*', tl):
            found[mot] = info
    return found

def _matcher_en(texte):
    """Retourne dict {cle_fr: info} des espèces trouvées dans un texte EN."""
    found = {}
    tl = texte.lower()
    for mot_en, cle_fr in _EN_TO_FR.items():
        if _re.search(r'\b' + _re.escape(mot_en.lower()) + r'\w*', tl):
            if cle_fr in _ESPECES_DB:
                found[cle_fr] = _ESPECES_DB[cle_fr]
    return found

def extraire_especes(passage, lang="fr", _stop_idx=None):
    """
    Extrait les espèces marines :
    - Si _stop_idx fourni : utilise les vrais chapitres du livre (méthode exhaustive)
    - Sinon : cherche dans le passage (compatibilité ascendante)
    Retourne du HTML formaté pour la Bootstrap Modal.
    """
    # ── Détection escale intermédiaire ───────────────────
    if passage and ("intermédiaire" in passage or "intermediate" in passage):
        if lang == "fr":
            return "<p>🗺️ Escale intermédiaire — aucun passage du roman associé.</p>"
        else:
            return "<p>🗺️ Intermediate stop — no novel passage associated.</p>"

    # ── Extraction par chapitres (méthode exhaustive) ────
    if _stop_idx is not None:
        texte_fr, texte_en = _get_passage_chapitres(_stop_idx)
        especes_fr = _matcher_fr(texte_fr)
        especes_en = _matcher_en(texte_en)
        especes = {**especes_fr, **especes_en}
        if lang == "fr":
            labels = [f"{v['emoji']} {v['fr']}" for v in especes.values()]
        else:
            labels = [f"{v['emoji']} {v['en']}" for v in especes.values()]
    else:
        # ── Fallback : cherche dans le passage fourni ────
        passage_lower = (passage or "").lower()
        labels = []
        if lang == "fr":
            for mot, info in _ESPECES_DB.items():
                if _re.search(r'\b' + _re.escape(mot) + r'[a-zéèêëàâùûüîïôœç]*', passage_lower):
                    labels.append(f"{info['emoji']} {info['fr']}")
        else:
            for mot_en, cle_fr in _EN_TO_FR.items():
                if mot_en in passage_lower and cle_fr in _ESPECES_DB:
                    labels.append(f"{_ESPECES_DB[cle_fr]['emoji']} {_ESPECES_DB[cle_fr]['en']}")

    if lang == "fr":
        intro = "<p style='color:#607080; font-size:0.85rem; margin-bottom:14px;'>Espèces identifiées dans l'extrait du roman :</p>"
        msg_vide = "<p>Aucune espèce marine identifiée dans ce passage.</p><p><em>Jules Verne décrit ici plutôt la géographie ou les événements du voyage.</em></p>"
    else:
        intro = "<p style='color:#607080; font-size:0.85rem; margin-bottom:14px;'>Species identified in the novel excerpt:</p>"
        msg_vide = "<p>No marine species identified in this passage.</p><p><em>Jules Verne describes geography or journey events here.</em></p>"

    if not labels:
        return msg_vide

    badges = "".join(
        f"<span style='display:inline-block; margin:4px; padding:5px 12px;"
        f"background:rgba(39,195,159,0.12); color:#27c39f;"
        f"border:1px solid rgba(39,195,159,0.3); border-radius:20px;"
        f"font-size:0.9rem;'>{e}</span>"
        for e in labels
    )
    return f"{intro}<div style='display:flex; flex-wrap:wrap; gap:4px;'>{badges}</div>"

# Conserver la compatibilité avec l'ancien code (ESPECES_FR / ESPECES_EN)
ESPECES_FR = {k: f"{v['emoji']} {v['fr']}" for k, v in _ESPECES_DB.items()}
ESPECES_EN = {mot_en: f"{_ESPECES_DB[cle_fr]['emoji']} {_ESPECES_DB[cle_fr]['en']}" for mot_en, cle_fr in _EN_TO_FR.items() if cle_fr in _ESPECES_DB}

print(f"✅ Fiche 2 — {len(_ESPECES_DB)} espèces dans le dictionnaire exhaustif")
print(f"   Chapitres FR P1 : {len(_ch_fr_p1)} | P2 : {len(_ch_fr_p2)} | EN : {len(_ch_en)}")

# =========================================================
# CONTENUS DÉTAILLÉS — FR ET EN SÉPARÉS
# Même structure que l'original : DETAILED_CONTENT[stop][lang][action]
# Les valeurs sont du HTML prêt à injecter dans la Bootstrap Modal
# =========================================================
MSG_INTER_FR = "<p>🗺️ Ce point est une escale de navigation intermédiaire, non décrite explicitement dans le roman de Jules Verne.</p>"
MSG_INTER_EN = "<p>🗺️ This is an intermediate navigation point, not explicitly described in Jules Verne's novel.</p>"


# =========================================================
# FICHE 3 — GÉNÉRATION IA LOCALE (OLLAMA + MISTRAL)
# Appelle Ollama sur localhost, génère le JSON une seule fois,
# le sauvegarde dans fiche3_science_19c.json pour Netlify.
# Ollama doit tourner en fond (icône lama dans la barre Windows).
# =========================================================

import urllib.request
import json as _json
from pathlib import Path

FICHE3_JSON = BASE_DIR / "fiche3_science_19c.json"
OLLAMA_URL  = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "mistral"

def _appeler_ollama(prompt):
    """Appelle Ollama en local et retourne le texte généré."""
    payload = _json.dumps({
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.4, "num_predict": 400}
    }).encode("utf-8")
    req = urllib.request.Request(
        OLLAMA_URL,
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST"
    )
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            result = _json.loads(resp.read().decode("utf-8"))
            return result.get("response", "").strip()
    except Exception as e:
        return f"[Ollama indisponible : {e}]"

def _construire_prompt_fr(nom_escale, passage_fr, event_fr):
    return f"""Tu es un expert en histoire des sciences du XIXe siècle et en littérature de Jules Verne.

Escale du Nautilus : {nom_escale}
Contexte de l'escale : {event_fr}
Extrait du roman (Jules Verne, 1870) :
"""{passage_fr[:800]}"""

En 3 à 4 paragraphes concis, décris les découvertes scientifiques réelles du XIXe siècle
qui correspondent à ce passage : savants de l'époque (Cuvier, Darwin, Lacépède, Faraday...),
disciplines (zoologie marine, électricité, géographie, chimie...), instruments,
expéditions scientifiques. Reste factuel et ancré dans le contexte 1866-1870.
Ne mentionne pas Jules Verne ni le Nautilus dans ta réponse.
Réponds directement en français, sans introduction ni conclusion."""

def _construire_prompt_en(nom_escale, passage_en, event_en):
    return f"""You are an expert in 19th-century history of science and Jules Verne's literature.

Nautilus stop: {nom_escale}
Context: {event_en}
Novel excerpt (Jules Verne, 1870):
"""{passage_en[:800]}"""

In 3 to 4 concise paragraphs, describe the real scientific discoveries of the 19th century
that relate to this passage: scientists of the era (Cuvier, Darwin, Lacépède, Faraday...),
disciplines (marine zoology, electricity, geography, chemistry...), instruments,
scientific expeditions. Stay factual and grounded in the 1866-1870 context.
Do not mention Jules Verne or the Nautilus in your answer.
Reply directly in English, without introduction or conclusion."""

def _html_fiche3_fr(nom_escale, texte_ia):
    paras = [p.strip() for p in texte_ia.split("\n") if p.strip()]
    paras_html = "".join(
        f"<p style='margin-bottom:12px; line-height:1.7; font-size:0.95rem;'>{p}</p>"
        for p in paras[:5]
    )
    return f"""
        <h3 style='color:#0a3d62;'>Découvertes scientifiques du XIXe siècle</h3>
        <h5 style='color:#27c39f; font-size:0.85rem; margin-bottom:14px;'>🔬 {nom_escale}</h5>
        <div style='color:#607080; font-size:0.8rem; margin-bottom:12px;'>
            ✨ Généré par Mistral (IA locale)
        </div>
        {paras_html}"""

def _html_fiche3_en(nom_escale, texte_ia):
    paras = [p.strip() for p in texte_ia.split("\n") if p.strip()]
    paras_html = "".join(
        f"<p style='margin-bottom:12px; line-height:1.7; font-size:0.95rem;'>{p}</p>"
        for p in paras[:5]
    )
    return f"""
        <h3 style='color:#0a3d62;'>19th-century scientific discoveries</h3>
        <h5 style='color:#27c39f; font-size:0.85rem; margin-bottom:14px;'>🔬 {nom_escale}</h5>
        <div style='color:#607080; font-size:0.8rem; margin-bottom:12px;'>
            ✨ Generated by Mistral (local AI)
        </div>
        {paras_html}"""

def generer_fiche3(df, force=False):
    """
    Génère fiche3_science_19c.json via Ollama.
    Si le fichier existe déjà, le charge sans recalculer (sauf si force=True).
    """
    if FICHE3_JSON.exists() and not force:
        print(f"✅ Fiche 3 déjà générée — chargement de {FICHE3_JSON.name}")
        return _json.loads(FICHE3_JSON.read_text(encoding="utf-8"))

    print(f"🤖 Génération fiche 3 via Ollama ({OLLAMA_MODEL})...")
    print(f"   ⚠️  Ollama doit tourner (icône lama dans la barre Windows)")
    print(f"   ⏱️  ~30 secondes par escale × {len(df)} escales — soyez patient !")

    # Test rapide de connexion
    test = _appeler_ollama("Réponds juste: OK")
    if "Ollama indisponible" in test:
        print(f"   ❌ Ollama non joignable : {test}")
        print(f"   → Lance Ollama depuis le menu Démarrer et relance cette cellule")
        return {}

    print(f"   ✅ Ollama répond — démarrage de la génération")
    fiche3 = {}

    for i, row in df.iterrows():
        stop_fr  = get_lang(row.get("Escales du Nautilus"), "fr") or f"Escale {i+1}"
        stop_en  = get_lang(row.get("Escales du Nautilus"), "en") or f"Stop {i+1}"
        event_fr = get_lang(row.get("Event"), "fr") or ""
        event_en = get_lang(row.get("Event"), "en") or ""
        passage_fr = get_passage(row, "fr") or event_fr
        passage_en = get_passage(row, "en") or event_en

        # Escales intermédiaires sans passage : contenu court
        if not passage_fr or "intermédiaire" in passage_fr:
            fiche3[str(i)] = {
                "fr": f"""
                    <h3 style='color:#0a3d62;'>Découvertes scientifiques du XIXe siècle</h3>
                    <h5 style='color:#27c39f; font-size:0.85rem; margin-bottom:14px;'>🔬 {stop_fr}</h5>
                    <p>🗺️ Escale de navigation intermédiaire — pas de passage du roman associé.</p>""",
                "en": f"""
                    <h3 style='color:#0a3d62;'>19th-century scientific discoveries</h3>
                    <h5 style='color:#27c39f; font-size:0.85rem; margin-bottom:14px;'>🔬 {stop_en}</h5>
                    <p>🗺️ Intermediate navigation stop — no novel passage associated.</p>"""
            }
            print(f"  [{i:>2}] ⏭️  {stop_fr[:35]:35s} (intermédiaire)")
            continue

        # Générer FR
        prompt_fr = _construire_prompt_fr(stop_fr, passage_fr, event_fr)
        texte_fr  = _appeler_ollama(prompt_fr)

        # Générer EN
        prompt_en = _construire_prompt_en(stop_en, passage_en, event_en)
        texte_en  = _appeler_ollama(prompt_en)

        fiche3[str(i)] = {
            "fr": _html_fiche3_fr(stop_fr, texte_fr),
            "en": _html_fiche3_en(stop_en, texte_en),
        }
        print(f"  [{i:>2}] ✅ {stop_fr[:35]:35s} — {len(texte_fr)} chars FR")

    # Sauvegarde
    FICHE3_JSON.write_text(
        _json.dumps(fiche3, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    print(f"\n✅ Fiche 3 sauvegardée : {FICHE3_JSON.resolve()}")
    print(f"   → Ajoute ce fichier à ton dépôt Git et déploie sur Netlify")
    return fiche3

# Lancement (met force=True pour régénérer même si le JSON existe déjà)
fiche3_content = generer_fiche3(df, force=False)
print(f"\n📦 {len(fiche3_content)} escales dans fiche3_content")


detailed_content = {}

for i, row in df.iterrows():
    stop_fr  = get_lang(row.get("Escales du Nautilus"), "fr") or f"Escale {i+1}"
    stop_en  = get_lang(row.get("Escales du Nautilus"), "en") or f"Stop {i+1}"
    event_fr = get_lang(row.get("Event"), "fr")
    event_en = get_lang(row.get("Event"), "en")

    passage_fr = get_passage(row, "fr")
    passage_en = get_passage(row, "en")

    # Points intermédiaires sans passage
    if not passage_fr and not passage_en:
        book_html_fr = f"<h3 style='color:#0a3d62;'>{stop_fr}</h3>{MSG_INTER_FR}"
        book_html_en = f"<h3 style='color:#0a3d62;'>{stop_en}</h3>{MSG_INTER_EN}"
    else:
        book_html_fr = make_html_content(
            stop_fr,
            passage_fr if passage_fr else "Extrait indisponible pour cette escale.",
            "📖 Extrait du livre"
        )
        book_html_en = make_html_content(
            stop_en,
            passage_en if passage_en else "Excerpt unavailable for this stop.",
            "📖 Book excerpt"
        )

    detailed_content[str(i)] = {
        # ── FRANÇAIS ──────────────────────────────────────────
        "fr": {
            "book_excerpt": book_html_fr,
            "marine_species": extraire_especes(passage_fr, "fr", _stop_idx=i),
            "science_19c": (
            fiche3_content.get(str(i), {}).get("fr")
            or f"""
                <h3 style='color:#0a3d62;'>Découvertes scientifiques du XIXe siècle</h3>
                <h5 style='color:#27c39f; font-size:0.85rem; margin-bottom:14px;'>🔬 {stop_fr}</h5>
                <p><strong>Contexte :</strong> {event_fr}</p>
                <p>Fiche 3 — relance le notebook pour générer.</p>"""
        ),
            "compare_1866_today": f"""
                <h3 style='color:#0a3d62;'>Comparaison entre 1866 et aujourd'hui</h3>
                <h5 style='color:#27c39f; font-size:0.85rem; margin-bottom:14px;'>⚗️ {stop_fr}</h5>
                <p>Fiche 4 — en cours de développement.</p>""",
        },
        # ── ENGLISH ───────────────────────────────────────────
        "en": {
            "book_excerpt": book_html_en,
            "marine_species": extraire_especes(passage_en, "en", _stop_idx=i),
            "science_19c": (
            fiche3_content.get(str(i), {}).get("en")
            or f"""
                <h3 style='color:#0a3d62;'>19th-century scientific discoveries</h3>
                <h5 style='color:#27c39f; font-size:0.85rem; margin-bottom:14px;'>🔬 {stop_en}</h5>
                <p><strong>Context:</strong> {event_en}</p>
                <p>Card 3 — rerun notebook to generate.</p>"""
        ),
            "compare_1866_today": f"""
                <h3 style='color:#0a3d62;'>Comparison between 1866 and today</h3>
                <h5 style='color:#27c39f; font-size:0.85rem; margin-bottom:14px;'>⚗️ {stop_en}</h5>
                <p>Card 4 — under development.</p>""",
        }
    }

print(f"✅ {len(detailed_content)} escales dans detailed_content")

# Vérification rapide des 3 premières
for k in ["0", "1", "2"]:
    fr = detailed_content[k]["fr"]["book_excerpt"][:80].replace("\n","")
    en = detailed_content[k]["en"]["book_excerpt"][:80].replace("\n","")
    print(f"  [{k}] 🇫🇷 {fr}...")
    print(f"  [{k}] 🇬🇧 {en}...")

# =========================================================
# POPUP BILINGUE — exactement le même format que l'original
# =========================================================
def make_popup(row, i):
    stop_fr  = html_module.escape(get_lang(row.get("Escales du Nautilus"), "fr") or f"Escale {i+1}")
    date_fr  = html_module.escape(get_lang(row.get("Date"),      "fr"))
    ocean_fr = html_module.escape(get_lang(row.get("Océan/Mer"), "fr"))
    event_fr = html_module.escape(get_lang(row.get("Event"),     "fr"))
    stop_en  = html_module.escape(get_lang(row.get("Escales du Nautilus"), "en") or f"Stop {i+1}")
    date_en  = html_module.escape(get_lang(row.get("Date"),      "en"))
    ocean_en = html_module.escape(get_lang(row.get("Océan/Mer"), "en"))
    event_en = html_module.escape(get_lang(row.get("Event"),     "en"))
    lat  = row["Latitude_Décimal"]
    lon  = row["Longitude_Décimal"]
    dist = f"{row['DistanceCentreTerre']:,}"

    return f"""
    <div style="display:flex; gap:20px; min-width:520px;">

        <!-- FRANÇAIS -->
        <div style="flex:1; border-right:1px solid #ccc; padding-right:10px;">
            <h4>Français 🇫🇷</h4>
            🗺️🧭<b>Escales du Nautilus :</b> {stop_fr}<br>
            📅 <b>Date :</b> {date_fr}<br>
            🌊 <b>Océan/Mer :</b> {ocean_fr}<br>
            📜 <b>Événement :</b> {event_fr}<br>
            ↕️ <b>Latitude :</b> {lat}<br>
            ↔️ <b>Longitude :</b> {lon}<br>
            🌍🧲 <b>Distance du centre de la Terre :</b> {dist} mètres
            <br><br>

            <div style="display:flex; gap:8px; flex-wrap:wrap;">
                <button class="submenu-toggle-btn btn btn-sm btn-primary"
                        data-menu-id="submenu-logbook-fr-{i}">
                    📖 Journal de bord vivant
                </button>
                <button class="submenu-toggle-btn btn btn-sm btn-outline-secondary"
                        data-menu-id="submenu-ai-fr-{i}">
                    ✨ Explorations IA
                </button>
            </div>

            <div id="submenu-logbook-fr-{i}" class="popup-submenu"
                 style="display:none; margin-top:10px; padding-top:6px;">
                <div style="display:flex; gap:6px; flex-wrap:wrap;">
                    <button class="submenu-action btn btn-sm btn-light"
                            data-stop="{i}" data-lang="fr" data-action="book_excerpt" data-group="logbook">
                        Extrait du livre
                    </button>
                    <button class="submenu-action btn btn-sm btn-light"
                            data-stop="{i}" data-lang="fr" data-action="marine_species" data-group="logbook">
                        Espèces maritimes mentionnées
                    </button>
                </div>
            </div>

            <div id="submenu-ai-fr-{i}" class="popup-submenu"
                 style="display:none; margin-top:10px; padding-top:6px;">
                <div style="display:flex; gap:6px; flex-wrap:wrap;">
                    <button class="submenu-action btn btn-sm btn-light"
                            data-stop="{i}" data-lang="fr" data-action="science_19c" data-group="ai">
                        Découvertes scientifiques du XIXe siècle
                    </button>
                    <button class="submenu-action btn btn-sm btn-light"
                            data-stop="{i}" data-lang="fr" data-action="compare_1866_today" data-group="ai">
                        Comparaison entre 1866 et aujourd'hui
                    </button>
                </div>
            </div>
        </div>

        <!-- ENGLISH -->
        <div style="flex:1; padding-left:10px;">
            <h4>English 🇬🇧</h4>
            🗺️🧭<b>Nautilus Stops :</b> {stop_en}<br>
            📅 <b>Date :</b> {date_en}<br>
            🌊 <b>Ocean/Sea :</b> {ocean_en}<br>
            📜 <b>Event :</b> {event_en}<br>
            ↕️ <b>Latitude :</b> {lat}<br>
            ↔️ <b>Longitude :</b> {lon}<br>
            🌍🧲 <b>Distance from Earth's center :</b> {dist} meters
            <br><br>

            <div style="display:flex; gap:8px; flex-wrap:wrap;">
                <button class="submenu-toggle-btn btn btn-sm btn-primary"
                        data-menu-id="submenu-logbook-en-{i}">
                    📖 Living Logbook
                </button>
                <button class="submenu-toggle-btn btn btn-sm btn-outline-secondary"
                        data-menu-id="submenu-ai-en-{i}">
                    ✨ AI Explorations
                </button>
            </div>

            <div id="submenu-logbook-en-{i}" class="popup-submenu"
                 style="display:none; margin-top:10px; padding-top:6px;">
                <div style="display:flex; gap:6px; flex-wrap:wrap;">
                    <button class="submenu-action btn btn-sm btn-light"
                            data-stop="{i}" data-lang="en" data-action="book_excerpt" data-group="logbook">
                        Book excerpt
                    </button>
                    <button class="submenu-action btn btn-sm btn-light"
                            data-stop="{i}" data-lang="en" data-action="marine_species" data-group="logbook">
                        Marine species mentioned
                    </button>
                </div>
            </div>

            <div id="submenu-ai-en-{i}" class="popup-submenu"
                 style="display:none; margin-top:10px; padding-top:6px;">
                <div style="display:flex; gap:6px; flex-wrap:wrap;">
                    <button class="submenu-action btn btn-sm btn-light"
                            data-stop="{i}" data-lang="en" data-action="science_19c" data-group="ai">
                        19th-century scientific discoveries
                    </button>
                    <button class="submenu-action btn btn-sm btn-light"
                            data-stop="{i}" data-lang="en" data-action="compare_1866_today" data-group="ai">
                        Comparison between 1866 and today
                    </button>
                </div>
            </div>
        </div>

    </div>"""

# =========================================================
# CARTE FOLIUM
# =========================================================
# =========================================================
# CARTE FOLIUM
# =========================================================
m = folium.Map(location=[20, -30], zoom_start=2, tiles="CartoDB positron")
marker_cluster = MarkerCluster().add_to(m)
coords = []

for i, row in df.iterrows():
    lat = row["Latitude_Décimal"]
    lon = row["Longitude_Décimal"]
    coords.append([lat, lon])
    tooltip = html_module.escape(get_lang(row.get("Escales du Nautilus"), "fr") or f"Escale {i+1}")

    # ── Marqueur DÉPART ──────────────────────────────────
    if i == 0:
        folium.Marker(
            location=[lat, lon],
            popup=folium.Popup(make_popup(row, i), max_width=620),
            tooltip=f"⚓ Départ — {tooltip}",
            icon=folium.Icon(color="green", icon="anchor", prefix="fa")
        ).add_to(m)  # hors cluster pour toujours le voir

    # ── Marqueur ARRIVÉE ─────────────────────────────────
    elif i == len(df) - 1:
        folium.Marker(
            location=[lat, lon],
            popup=folium.Popup(make_popup(row, i), max_width=620),
            tooltip=f"🏁 Arrivée — {tooltip}",
            icon=folium.Icon(color="red", icon="flag", prefix="fa")
        ).add_to(m)  # hors cluster pour toujours le voir

    # ── Marqueurs normaux ────────────────────────────────
    else:
        folium.Marker(
            location=[lat, lon],
            popup=folium.Popup(make_popup(row, i), max_width=620),
            tooltip=tooltip,
            icon=folium.Icon(color="blue", icon="info-sign")
        ).add_to(marker_cluster)

# ── Ligne bleue du trajet ─────────────────────────────
if len(coords) > 1:
    folium.PolyLine(
        coords,
        color="darkblue",
        weight=3,
        opacity=0.75
    ).add_to(m)

print(f"✅ {len(coords)} marqueurs ajoutés — départ ⚓ vert · arrivée 🏁 rouge · trajet bleu")

# =========================================================
# INJECTION : Loader + Bootstrap Modal + JS
# Exactement le même système que ton original qui fonctionne
# =========================================================
custom_html = f"""
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Michroma&display=swap" rel="stylesheet">

<!-- Loader -->
<div id="dataLoaderOverlay" style="
    display:none; position:fixed; inset:0;
    background:rgba(0,0,0,0.32); z-index:2000;
    align-items:center; justify-content:center;
    backdrop-filter:blur(2px);">
    <div style="
        padding:18px 24px; border-radius:14px;
        background:rgba(12,12,16,0.94); color:#27c39f;
        font-size:16px; font-weight:600;
        box-shadow:0 8px 24px rgba(0,0,0,0.25);
        border:1px solid rgba(39,195,159,0.22);
        font-family:Arial,sans-serif;">
        Traitement des données en cours<span class="loading-dots"></span>
    </div>
</div>

<!-- Bootstrap Modal -->
<div class="modal fade" id="logbookModal" tabindex="-1"
     aria-labelledby="logbookModalLabel" aria-hidden="true">
  <div class="modal-dialog modal-lg modal-dialog-centered">
    <div class="modal-content" style="border-radius:16px; overflow:hidden;">
      <div class="modal-header" style="background:#0a0e14; border-bottom:1px solid rgba(39,195,159,0.2);">
        <h5 class="modal-title" id="logbookModalLabel"
            style="color:#27c39f; font-family:'Michroma',sans-serif; letter-spacing:0.05em;">
            Journal de bord vivant
        </h5>
        <button type="button" class="btn-close btn-close-white"
                data-bs-dismiss="modal" aria-label="Close"></button>
      </div>
      <div class="modal-body" id="logbookModalBody"
           style="font-size:1rem; line-height:1.65; background:#0f1923; color:#c8d8e8;
                  font-family:Arial,sans-serif; padding:24px;">
      </div>
    </div>
  </div>
</div>

<style>
.leaflet-popup-content {{ min-width:520px !important; }}
.popup-submenu .btn {{ font-size:12px; }}
.loading-dots::after {{
    content:""; display:inline-block; width:1.6em; text-align:left;
    animation:dotsSteps 1.4s steps(4,end) infinite;
}}
@keyframes dotsSteps {{
    0%   {{ content:""; }}
    25%  {{ content:"."; }}
    50%  {{ content:".."; }}
    75%  {{ content:"..."; }}
    100% {{ content:""; }}
}}
</style>

<script>
const DETAILED_CONTENT = {json.dumps(detailed_content, ensure_ascii=False)};

function showLoader() {{
    const l = document.getElementById("dataLoaderOverlay");
    if (l) l.style.display = "flex";
}}
function hideLoader() {{
    const l = document.getElementById("dataLoaderOverlay");
    if (l) l.style.display = "none";
}}
function openModalWithContent(title, content) {{
    document.getElementById("logbookModalLabel").innerHTML = title;
    document.getElementById("logbookModalBody").innerHTML  = content;
    const modal = new bootstrap.Modal(document.getElementById("logbookModal"));
    modal.show();
}}

document.addEventListener("click", function(e) {{

    // Bouton toggle (Journal de bord / Explorations IA)
    const toggleBtn = e.target.closest(".submenu-toggle-btn");
    if (toggleBtn) {{
        const menuId = toggleBtn.getAttribute("data-menu-id");
        const menu   = document.getElementById(menuId);
        if (menu) {{
            menu.style.display = (menu.style.display === "none" || menu.style.display === "")
                ? "block" : "none";
        }}
        return;
    }}

    // Bouton action (Extrait du livre, Espèces, etc.)
    const actionBtn = e.target.closest(".submenu-action");
    if (actionBtn) {{
        const stop   = actionBtn.getAttribute("data-stop");
        const lang   = actionBtn.getAttribute("data-lang") || "fr";
        const action = actionBtn.getAttribute("data-action");
        const group  = actionBtn.getAttribute("data-group");

        const titles = {{
            fr: {{
                book_excerpt:       "📖 Extrait du livre",
                marine_species:     "🐙 Espèces maritimes mentionnées",
                science_19c:        "🔬 Découvertes scientifiques du XIXe siècle",
                compare_1866_today: "⚗️ Comparaison entre 1866 et aujourd'hui"
            }},
            en: {{
                book_excerpt:       "📖 Book excerpt",
                marine_species:     "🐙 Marine species mentioned",
                science_19c:        "🔬 19th-century scientific discoveries",
                compare_1866_today: "⚗️ Comparison between 1866 and today"
            }}
        }};

        const fallback = lang === "fr"
            ? "<p>Contenu disponible bientôt.</p>"
            : "<p>Content coming soon.</p>";

        const content = (
            DETAILED_CONTENT[stop] &&
            DETAILED_CONTENT[stop][lang] &&
            DETAILED_CONTENT[stop][lang][action]
        ) ? DETAILED_CONTENT[stop][lang][action] : fallback;

        if (group === "ai") {{
            showLoader();
            setTimeout(() => {{
                hideLoader();
                openModalWithContent(titles[lang][action] || "Exploration IA", content);
            }}, 2400);
        }} else {{
            openModalWithContent(titles[lang][action] || "Journal de bord vivant", content);
        }}
        return;
    }}
}});
</script>
"""

m.get_root().html.add_child(folium.Element(custom_html))

fix_map = """
<script>
window.addEventListener('load', function() {
    setTimeout(function() {
        var mapId = Object.keys(window).filter(k => k.startsWith('map_'))[0];
        if (window[mapId]) {
            window[mapId].invalidateSize();
            window[mapId].setView([20, -30], 2);
        }
    }, 300);
});
</script>
"""
m.get_root().html.add_child(folium.Element(fix_map))

# =========================================================
# EXPORT
# =========================================================
m.save(str(OUTPUT_HTML))
print(f"✅ Fichier généré : {OUTPUT_HTML.resolve()}")
print()
print("📋 Prochaine étape :")
print(f"   → Copie '{OUTPUT_HTML.name}' dans ton projet VSCode")
print(f"   → Remplace 'carte_nautilus_multilang14.html'")
print(f"   → Deploy sur Netlify")

display(HTML(f'<a href="{OUTPUT_HTML.name}" target="_blank">🔗 Ouvrir la carte HTML générée</a>'))  





SyntaxError: invalid syntax (3008347299.py, line 457)